In [ ]:
!pip install -q transformers datasets evaluate accelerate kaggle scikit-learn seaborn

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd

# Load the dataset (adjust the filename if necessary based on your unzipped file)
# Common filenames for IMDB datasets are 'IMDB Dataset.csv' or 'movie_reviews.csv'
try:
    df = pd.read_csv('IMDB Dataset.csv')
except FileNotFoundError:
    print("IMDB Dataset.csv not found. Please check the unzipped file name and path.")
    print("Listing files in current directory:")
    !ls -l
    df = None

if df is not None:
    print("Dataset loaded successfully. Displaying the first 5 rows:")
    display(df.head())
    print("\nDataset Information:")
    df.info()
    print("\nMissing values:")
    print(df.isnull().sum())

In [ ]:
import re

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

if df is not None:
    print("Cleaning text data...")
    df['review'] = df['review'].apply(clean_text)
    print("Text cleaning complete. Displaying first 5 rows of cleaned data:")
    display(df.head())
else:
    print("DataFrame not loaded. Cannot perform text cleaning.")

In [ ]:
from sklearn.model_selection import train_test_split

if df is not None:
    # Split data into training and temporary (validation + test) sets
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

    # Split temporary set into validation and test sets
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['sentiment'])

    print(f"Training set size: {len(train_df)}")
    print(f"Validation set size: {len(val_df)}")
    print(f"Test set size: {len(test_df)}")

    # Display sentiment distribution in each set to confirm stratification
    print("\nSentiment distribution in training set:")
    print(train_df['sentiment'].value_counts(normalize=True))
    print("\nSentiment distribution in validation set:")
    print(val_df['sentiment'].value_counts(normalize=True))
    print("\nSentiment distribution in test set:")
    print(test_df['sentiment'].value_counts(normalize=True))
else:
    print("DataFrame not loaded. Cannot perform data splitting.")

In [ ]:
from transformers import BertTokenizer
import torch

# Initialize the BERT tokenizer
print("Loading tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("Tokenizer loaded.")

# Function to tokenize the reviews
def tokenize_function(examples):
    # 'examples['review']' is already a list of strings when batched=True
    return tokenizer(examples['review'], padding="max_length", truncation=True, max_length=128)

# Convert DataFrames to Hugging Face Datasets
from datasets import Dataset

print("Converting DataFrames to Hugging Face Datasets...")
# Map sentiment labels to integers (0 for negative, 1 for positive)
label_map = {'negative': 0, 'positive': 1}
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label'] = val_df['sentiment'].map(label_map)
test_df['label'] = test_df['sentiment'].map(label_map)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
print("DataFrames converted.")

print("Tokenizing datasets...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
print("Datasets tokenized.")

# Remove original text and sentiment columns, keep only tokenized inputs and labels
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["review", "sentiment"])
tokenized_val_dataset = tokenized_val_dataset.remove_columns(["review", "sentiment"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["review", "sentiment"])

# Set the format to PyTorch tensors
tokenized_train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])
tokenized_val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])
tokenized_test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])

print("Tokenization complete. Displaying first tokenized example from training set:")
print(tokenized_train_dataset[0])

In [ ]:
from transformers import AutoModelForSequenceClassification

print("Loading pre-trained BERT model for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
print("Model loaded.")

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from evaluate import load
import os

# Set the TensorBoard logging directory environment variable
os.environ['TENSORBOARD_LOGGING_DIR'] = './logs'

# Define metrics for evaluation
def compute_metrics(eval_pred):
    load_accuracy = load("accuracy")
    load_f1 = load("f1")
    load_precision = load("precision")
    load_recall = load("recall")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    precision = load_precision.compute(predictions=predictions, references=labels)["precision"]
    recall = load_recall.compute(predictions=predictions, references=labels)["recall"]
    return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}

# Configure training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    # logging_dir='./logs', # Deprecated - handled by TENSORBOARD_LOGGING_DIR env var
    logging_steps=100,
    save_strategy="epoch",
    metric_for_best_model="f1",
    report_to="tensorboard"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    compute_metrics=compute_metrics,
)

# Train the model
print("Starting model training...")
trainer.train()
print("Model training complete.")

In [ ]:
print("Evaluating model on test set...")
test_results = trainer.evaluate(tokenized_test_dataset)
print("Model evaluation complete. Test Set Metrics:")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("Generating predictions on the test set...")
predictions_output = trainer.predict(tokenized_test_dataset)
predictions = np.argmax(predictions_output.predictions, axis=-1)
references = predictions_output.label_ids

# Compute Confusion Matrix
cm = confusion_matrix(references, predictions, labels=[0, 1]) # 0: negative, 1: positive

# Display Confusion Matrix
print("Displaying Confusion Matrix:")
fig, ax = plt.subplots(figsize=(8, 8))
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negative', 'positive'])
display_cm.plot(cmap=plt.cm.Blues, ax=ax)
plt.title("Confusion Matrix for Sentiment Analysis")
plt.show()

## **8. Conclusion: Model Performance Summary**

The fine-tuned BERT model achieved strong performance on the IMDB sentiment analysis task, as evidenced by the evaluation on the unseen test set.

### **Test Set Metrics:**
*   **Loss**: 0.4125
*   **Accuracy**: 0.9060
*   **F1 Score**: 0.9067
*   **Precision**: 0.8996
*   **Recall**: 0.9140

These metrics indicate that the model is highly effective in classifying movie reviews as either positive or negative, achieving over 90% in accuracy, F1-score, precision, and recall. The F1-score, which balances precision and recall, is particularly strong, suggesting a robust model that effectively identifies both positive and negative sentiments without significant bias towards one class.

### **Confusion Matrix Analysis:**
*   **True Positives (Positive reviews correctly identified)**: 2285
*   **True Negatives (Negative reviews correctly identified)**: 2245
*   **False Positives (Negative reviews incorrectly identified as Positive)**: 255
*   **False Negatives (Positive reviews incorrectly identified as Negative)**: 215

The confusion matrix further confirms the model's balanced performance, with a low number of false positives and false negatives relative to the true classifications. This demonstrates the model's ability to generalize well to new, unseen review data, effectively leveraging transfer learning from the pre-trained BERT model for this specific sentiment classification task.